# 5. Pair exercise: the tumour board needs answers in 30 minutes

You and your partner are the analytics team of a breast cancer unit. Tomorrow the **tumour board** (surgeons, oncologists, nurses, a patient representative) meets to review its follow-up policy after surgery. They have sent you four questions:

1. How quickly does breast cancer come back after surgery, and what can we tell patients at 1, 3 and 5 years?
2. Does hormone therapy reduce recurrence, and by how much?
3. Which patients are at highest risk, once we account for everything at the same time?
4. Can a model help us decide who needs closer follow-up?

You have a real trial dataset, 7 ready-made plots and the tools from notebooks 01 and 02: Kaplan-Meier (KM) curves, log-rank tests, restricted mean survival time (RMST) and the Cox proportional hazards (CPH) model

**Timing (about 30 minutes):** setup 3 min, Plots 1 to 7 about 3 min each, tumour board brief 5 min. If you finish early, use the "Your turn" section to test or contrast your findings with extra plots

## Setup
Run the two cells below. No edits needed. The second cell defines small helper functions you can reuse in the "Your turn" section

In [ ]:
# Setup
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from lifelines import KaplanMeierFitter, CoxPHFitter
from lifelines.datasets import load_gbsg2
from lifelines.statistics import logrank_test, multivariate_logrank_test, proportional_hazard_test
from lifelines.plotting import add_at_risk_counts
from lifelines.utils import restricted_mean_survival_time, concordance_index
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.precision", 3)

HORIZONS = [1, 3, 5]   # years: the time points the tumour board cares about
T_MAX = 7              # x-axis limit in years (maximum follow-up is about 7.3 years)

# Colour-blind safe colours (Okabe-Ito)
C = {"blue": "#0072B2", "orange": "#E69F00", "red": "#D55E00", "green": "#009E73",
     "purple": "#CC79A7", "grey": "#7F7F7F", "sky": "#56B4E9"}

In [ ]:
# Helper functions (no need to edit, reuse them in "Your turn")

def fmt_p(p):
    # Readable p-value for plot titles
    return "p < 0.001" if p < 0.001 else f"p = {p:.3f}"


def km_panel(ax, data, group_col, labels=None, colors=None, title=None,
             at_risk=False, ci=True, horizon_lines=True):
    # Plot one KM curve per group on ax, run a log-rank test and return (p_value, fitters)
    labels = labels or {g: str(g) for g in sorted(data[group_col].dropna().unique())}
    colors = colors or {}
    fitters = []
    for g, lab in labels.items():
        sub = data.loc[data[group_col] == g]
        if len(sub) < 5:
            continue
        kmf = KaplanMeierFitter(label=f"{lab} (n={len(sub)})")
        kmf.fit(sub["years"], sub["event"])
        kmf.plot_survival_function(ax=ax, ci_show=ci, color=colors.get(g), linewidth=2)
        fitters.append(kmf)
    sub = data.dropna(subset=[group_col])
    p = multivariate_logrank_test(sub["years"], sub[group_col], sub["event"]).p_value
    if horizon_lines:
        for h in HORIZONS:
            ax.axvline(h, color="lightgrey", linestyle=":", linewidth=1, zorder=0)
    ax.set_xlim(0, T_MAX)
    ax.set_ylim(0, 1.02)
    ax.set_xlabel("Years since surgery")
    ax.set_ylabel("Recurrence-free survival")
    ax.set_title(f"{title or group_col}  [log-rank {fmt_p(p)}]", fontsize=11)
    ax.legend(loc="lower left", fontsize=9)
    if at_risk:
        add_at_risk_counts(*fitters, ax=ax, rows_to_show=["At risk"])
    return p, fitters


def km_summary(data, group_col, labels=None, t_rmst=5):
    # Table with n, events, RFS at 1/3/5 years and RMST to t_rmst years per group
    labels = labels or {g: str(g) for g in sorted(data[group_col].dropna().unique())}
    rows = []
    for g, lab in labels.items():
        sub = data.loc[data[group_col] == g]
        kmf = KaplanMeierFitter().fit(sub["years"], sub["event"])
        row = {"group": lab, "n": len(sub), "events": int(sub["event"].sum())}
        for h in HORIZONS:
            row[f"RFS {h} y"] = float(kmf.predict(h))
        row[f"RMST to {t_rmst} y [years]"] = float(restricted_mean_survival_time(kmf, t=t_rmst))
        rows.append(row)
    return pd.DataFrame(rows).set_index("group").round(3)


def quick_km(col, bins=None, labels=None, title=None, data=None):
    # One-line KM comparison for exploration: quick_km("age", bins=[0, 40, 60, 100])
    d = (data if data is not None else df).copy()
    if bins is not None:
        d[col + "_band"] = pd.cut(d[col], bins=bins, labels=labels)
        col = col + "_band"
    fig, ax = plt.subplots(figsize=(7, 5))
    km_panel(ax, d, col, title=title or col, at_risk=True)
    plt.tight_layout()
    plt.show()
    display(km_summary(d, col))


def quick_cox(covariates, data=None, strata=None):
    # Fit a Cox model on any list of numeric columns and show hazard ratios
    d = (data if data is not None else df)
    cols = covariates + ["years", "event"] + (strata or [])
    m = CoxPHFitter().fit(d[cols], duration_col="years", event_col="event", strata=strata)
    out = m.summary[["exp(coef)", "exp(coef) lower 95%", "exp(coef) upper 95%", "p"]]
    out.columns = ["HR", "HR low 95%", "HR high 95%", "p"]
    display(out.round(3))
    print(f"C-index (in-sample): {m.concordance_index_:.3f}")
    return m

## The data: German Breast Cancer Study Group 2 (GBSG2) trial

- **Who:** 686 women with primary node-positive breast cancer (cancer found in the lymph nodes of the armpit), treated with surgery and chemotherapy in a randomised 2×2 trial in Germany ([Schumacher et al., 1994](https://search.r-project.org/CRAN/refmans/TH.data/html/GBSG2.html))
- **Time origin:** surgery. **Event:** recurrence of the cancer or death, whichever came first, called recurrence-free survival (RFS)
- **Censoring:** patient still recurrence-free at her last visit, mostly because the study ended (patients joined over several years)
- The dataset ships with `lifelines`, so no download is needed

| Variable | Meaning | Unit |
|---|---|---|
| `hormone_therapy` | Received hormone therapy (tamoxifen), 1 = yes | binary |
| `age` | Age at surgery | years |
| `postmenopausal` | Menopausal status, 1 = postmenopausal | binary |
| `tsize` | Tumour size | mm |
| `tgrade` | Tumour grade (I = cells look almost normal, III = very abnormal, aggressive) | I / II / III |
| `pnodes` | Number of lymph nodes with cancer | count |
| `progrec`, `estrec` | Progesterone (PR) and estrogen (ER) receptor level | fmol |
| `pr_positive`, `er_positive` | Receptor level ≥ 10 fmol, the classical "receptor-positive" cut-off | binary |
| `years`, `event` | Follow-up time and event flag (1 = recurrence or death, 0 = censored) | years, binary |

In [ ]:
# Load and prepare the data
raw = load_gbsg2()

df = raw.assign(
    years=raw["time"] / 365.25,                        # days -> years
    event=raw["cens"].astype(int),                     # 1 = recurrence or death, 0 = censored
    hormone_therapy=(raw["horTh"] == "yes").astype(int),
    postmenopausal=(raw["menostat"] == "Post").astype(int),
    pr_positive=(raw["progrec"] >= 10).astype(int),
    er_positive=(raw["estrec"] >= 10).astype(int),
    nodes_band=pd.cut(raw["pnodes"], bins=[0, 3, 9, 100], labels=["1-3", "4-9", "10+"]),
)

print(f"Patients: {len(df)} | events: {df['event'].sum()} ({df['event'].mean():.0%}) | "
      f"censored: {(1 - df['event']).sum()} ({1 - df['event'].mean():.0%})")
print(f"Observed time: median {df['years'].median():.1f} years, max {df['years'].max():.1f} years")
df.head()

## Plot 1. The big picture: how fast does cancer come back?

**Place your bet:** what share of women will be recurrence-free 5 years after surgery? (a) 80% (b) 65% (c) 50% (d) 35%

**What the plot shows**
- Left: when follow-up ended for each patient, split by event (recurrence or death) vs censored
- Right: KM curve for the whole cohort, with the number of women still at risk below the axis

In [ ]:
# Plot 1. Follow-up structure and overall KM curve
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5), gridspec_kw={"width_ratios": [1, 1.3]})

# (a) When did follow-up end, and how?
sns.histplot(data=df.assign(status=df["event"].map({1: "Event (recurrence or death)", 0: "Censored"})),
             x="years", hue="status", multiple="stack", bins=28,
             palette={"Event (recurrence or death)": C["red"], "Censored": C["grey"]}, ax=axes[0])
axes[0].set_title("(a) End of follow-up: event vs censored", fontsize=11)
axes[0].set_xlabel("Years since surgery")
axes[0].set_ylabel("Number of patients")

# (b) Overall KM curve
kmf_all = KaplanMeierFitter(label="All patients").fit(df["years"], df["event"])
kmf_all.plot_survival_function(ax=axes[1], color=C["blue"], linewidth=2)
for h in HORIZONS:
    s = float(kmf_all.predict(h))
    axes[1].plot(h, s, "o", color=C["blue"])
    axes[1].annotate(f"{s:.0%}", (h, s), textcoords="offset points", xytext=(6, 8), fontsize=10)
med = kmf_all.median_survival_time_
axes[1].axhline(0.5, color=C["grey"], linestyle="--", linewidth=1)
axes[1].axvline(med, color=C["grey"], linestyle="--", linewidth=1)
axes[1].annotate(f"median {med:.1f} y", (med, 0.5), textcoords="offset points", xytext=(6, -16), fontsize=10)
axes[1].set_xlim(0, T_MAX)
axes[1].set_ylim(0, 1.02)
axes[1].set_title("(b) Kaplan-Meier recurrence-free survival", fontsize=11)
axes[1].set_xlabel("Years since surgery")
axes[1].set_ylabel("Recurrence-free survival")
add_at_risk_counts(kmf_all, ax=axes[1], rows_to_show=["At risk", "Censored", "Events"])
plt.tight_layout()
plt.show()

# Numbers behind the plot
ci = kmf_all.confidence_interval_survival_function_
rows = []
for h in HORIZONS:
    lo, hi = ci.loc[:h].iloc[-1]
    rows.append({"horizon [years]": h, "RFS": float(kmf_all.predict(h)), "95% CI low": lo, "95% CI high": hi})
display(pd.DataFrame(rows).round(3))
print(f"Median RFS: {med:.2f} years | RMST to 5 years: {restricted_mean_survival_time(kmf_all, t=5):.2f} years")

### Plot 1 (continued). Follow-up of each woman

**What the plot shows:** one line per woman, from the start of her follow-up to its end
- **x** = recurrence or death happened (the event)
- **?** = censored: she was still recurrence-free at her last visit, so we do not know what happens next
- **(a) Illustration with invented women:** women have surgery on different calendar dates (staggered entry) but the study stops on one date, so the women who joined late are censored after a short follow-up even though they are well
- **(b) Real trial data:** the same idea after "shifting the data to the left", so every line starts at the woman's own surgery (time 0). The dataset only records time since surgery, so it has no calendar dates

In [ ]:
# Plot 1 (continued). Follow-up of individual women: calendar time vs time since surgery
from matplotlib.lines import Line2D

def lifeline_plot(ax, start, stop, event):
    # One horizontal line per woman, from the start of her follow-up to its end
    # x = recurrence or death happened, ? = censored (still recurrence-free at her last visit)
    for y, (s, e, ev) in enumerate(zip(start, stop, event)):
        ax.plot([s, e], [y, y], color="black", linewidth=1)
        ax.scatter(e, y, marker="x" if ev else "$?$", s=70 if ev else 110,
                   color=C["red"] if ev else C["grey"], zorder=3)
    ax.set_yticks([])
    ax.set_ylim(len(stop) - 0.4, -0.6)
    ax.set_xlim(-0.2, 7.4)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# (a) ILLUSTRATION with 6 invented women: everyone has surgery on a different calendar date (staggered entry)
entry = np.array([0.0, 0.5, 1.0, 2.0, 3.5, 4.5])        # calendar year of surgery
time_to_event = np.array([2.5, 6.0, 1.5, 4.5, 3.5, 1.0])  # years until recurrence or death, if followed long enough
study_end = 6.0                                           # the study stops on one calendar date for everybody
stop = np.minimum(entry + time_to_event, study_end)
event_a = (entry + time_to_event <= study_end).astype(int)
lifeline_plot(axes[0], entry, stop, event_a)
axes[0].axvline(study_end, color=C["blue"], linestyle="--", linewidth=1.5)
axes[0].text(study_end + 0.1, 2.5, "Study ends\n(same date\nfor everyone)", color=C["blue"], va="center", fontsize=10)
axes[0].set_title("(a) Illustration, 6 invented women: CALENDAR time", fontsize=11)
axes[0].set_xlabel("Calendar years since the trial started")
axes[0].set_ylabel("One line per woman")

# (b) REAL data: 15 women (7 events + 8 censored, close to the real 44% / 56% mix), every line starts at her own surgery
# (time 0 = "shifted to the left"). The dataset only has time since surgery, so it has no calendar dates
sample = pd.concat([df[df["event"] == 1].sample(7, random_state=RANDOM_STATE),
                    df[df["event"] == 0].sample(8, random_state=RANDOM_STATE)]).sort_values("years", ascending=False)
lifeline_plot(axes[1], np.zeros(len(sample)), sample["years"], sample["event"])
axes[1].set_title("(b) Real trial data, 15 women: TIME SINCE SURGERY", fontsize=11)
axes[1].set_xlabel("Years since surgery")

axes[1].legend(handles=[
    Line2D([], [], marker="x", color=C["red"], linestyle="", markersize=8, label="x  Recurrence or death (event)"),
    Line2D([], [], marker="$?$", color=C["grey"], linestyle="", markersize=10, label="?  Censored: still well at last visit"),
], loc="lower right", fontsize=9, frameon=True)
plt.tight_layout()
plt.show()
print(f"Panel (b) sample: {int(sample['event'].sum())} events, {int((1 - sample['event']).sum())} censored")

**Questions**
1. Was your bet right? What would you tell a patient about her chance of being recurrence-free at 1, 3 and 5 years?
2. In which period is the risk of recurrence concentrated? Look at where the curve is steepest
3. **Compare with the ICU notebook:** there, censoring was "discharged alive" and made 1 − KM misleading. What causes censoring here (look at panel a and at the plot of individual women)? Can we read 1 − KM as "probability of recurrence or death by year t"? Why?
4. RMST to 5 years is about 3.6 years. Translate this into one sentence a patient would understand

**Your notes**
-

## Plot 2. Does hormone therapy help?

**Place your bet:** among 100 women, how many more will be recurrence-free at 5 years with hormone therapy? (a) 0 to 2 (b) about 5 (c) about 15 (d) more than 25

**What the plot shows:** KM curves for women with and without hormone therapy. The shaded area between the curves up to 5 years is the **RMST difference**: extra recurrence-free time gained, on average, within 5 years

In [ ]:
# Plot 2. KM by hormone therapy with the RMST gap shaded
ht_labels = {0: "No hormone therapy", 1: "Hormone therapy"}
ht_colors = {0: C["grey"], 1: C["green"]}

fig, ax = plt.subplots(figsize=(9, 6.5))
p_ht, (km_no, km_yes) = km_panel(ax, df, "hormone_therapy", labels=ht_labels, colors=ht_colors,
                                  title="Recurrence-free survival by hormone therapy", at_risk=True,
                                  ci=False)

# Shade the area between the curves up to 5 years (= RMST difference)
grid = np.linspace(0, 5, 500)
s_yes, s_no = km_yes.predict(grid).to_numpy(), km_no.predict(grid).to_numpy()
ax.fill_between(grid, s_no, s_yes, color=C["green"], alpha=0.15, step="post", label="RMST gap to 5 y")
ax.legend(loc="lower left", fontsize=9)
plt.tight_layout()
plt.show()

summary_ht = km_summary(df, "hormone_therapy", labels=ht_labels)
display(summary_ht)
gain_5y = summary_ht.loc["Hormone therapy", "RFS 5 y"] - summary_ht.loc["No hormone therapy", "RFS 5 y"]
gap_rmst = summary_ht.iloc[1, -1] - summary_ht.iloc[0, -1]
print(f"Absolute gain in RFS at 5 years: {gain_5y:+.1%} | "
      f"RMST gain to 5 years: {gap_rmst:.2f} years (about {gap_rmst * 12:.0f} months)")

**Questions**
1. Was your bet right? Express the effect in three ways: absolute difference at 5 years, RMST gain in months, and "number needed to treat" (NNT = 1 / absolute difference)
2. When do the curves start to separate? Is the benefit immediate or does it build up over time?
3. **Skeptic's turn:** is this comparison fair? What would need to be true for this difference to be caused by the therapy? (Hint: Plot 4)

**Your notes**
-

## Plot 3. Who is at higher risk? One factor at a time

**Place your bet:** which of these four factors will separate the curves the most? Nodes, grade, progesterone receptor (PR) status, menopausal status

**What the plot shows:** four KM comparisons, each with its log-rank p-value in the title. Remember from notebook 01: the p-value says whether curves differ, the plot shows **how much** and **when**

In [ ]:
# Plot 3. Prognostic factors, one at a time (small multiples)
panels = [
    ("nodes_band", {"1-3": "1-3 nodes", "4-9": "4-9 nodes", "10+": "10+ nodes"},
     {"1-3": C["sky"], "4-9": C["orange"], "10+": C["red"]}, "(a) Positive lymph nodes"),
    ("tgrade", {"I": "Grade I", "II": "Grade II", "III": "Grade III"},
     {"I": C["sky"], "II": C["orange"], "III": C["red"]}, "(b) Tumour grade"),
    ("pr_positive", {1: "PR positive", 0: "PR negative"},
     {1: C["sky"], 0: C["red"]}, "(c) Progesterone receptor (PR)"),
    ("postmenopausal", {0: "Premenopausal", 1: "Postmenopausal"},
     {0: C["sky"], 1: C["purple"]}, "(d) Menopausal status"),
]

fig, axes = plt.subplots(2, 2, figsize=(14, 10), sharey=True)
tables = []
for ax, (col, labels, colors, title) in zip(axes.ravel(), panels):
    km_panel(ax, df, col, labels=labels, colors=colors, title=title, ci=True)
    t = km_summary(df, col, labels=labels)
    t.insert(0, "factor", title[4:])
    tables.append(t)
plt.tight_layout()
plt.show()

display(pd.concat(tables)[["factor", "n", "events", "RFS 5 y", "RMST to 5 y [years]"]])

**Questions**
1. Rank the four factors by how much they separate the curves. Use the 5-year RFS gap, not only the p-value
2. Look at grade: is the step from I to II similar to the step from II to III? What does that suggest?
3. Menopausal status shows no difference here. Does that mean it is irrelevant? Keep it in mind for Plot 4
4. Which factor would you put on a simple "high-risk" checklist for the follow-up clinic, and why?

**Your notes**
-

## Plot 4. Who actually received hormone therapy? (the fairness check)

**Swap roles now:** Driver becomes Skeptic

**Place your bet:** were treated and untreated women similar at baseline? (yes / no / not sure)

**What the plot shows**
- Left: share of women who received hormone therapy, by menopausal status
- Middle and right: the hormone therapy comparison **within** each menopausal group (a stratified comparison)
- The table below the plot is a classic "Table 1": baseline characteristics by treatment group

In [ ]:
# Plot 4. Baseline balance and stratified comparison
fig, axes = plt.subplots(1, 3, figsize=(17, 5.5), gridspec_kw={"width_ratios": [0.7, 1, 1]})

# (a) Share treated by menopausal status
share = df.groupby("postmenopausal")["hormone_therapy"].agg(["mean", "size"])
bars = axes[0].bar(["Premenopausal", "Postmenopausal"], share["mean"], color=[C["sky"], C["purple"]])
for b, (m, n) in zip(bars, share.to_numpy()):
    axes[0].text(b.get_x() + b.get_width() / 2, m + 0.01, f"{m:.0%}\n(n={int(n)})", ha="center", fontsize=10)
axes[0].set_ylim(0, 0.7)
axes[0].set_ylabel("Share receiving hormone therapy")
axes[0].set_title("(a) Who got hormone therapy?", fontsize=11)

# (b, c) Hormone therapy effect within each menopausal group
for ax, (meno, name) in zip(axes[1:], [(0, "Premenopausal"), (1, "Postmenopausal")]):
    km_panel(ax, df.loc[df["postmenopausal"] == meno], "hormone_therapy",
             labels=ht_labels, colors=ht_colors, title=f"({'b' if meno == 0 else 'c'}) {name} women only")
plt.tight_layout()
plt.show()

# Table 1: baseline characteristics by treatment group
table1 = df.groupby("hormone_therapy").agg(
    n=("age", "size"),
    age_median=("age", "median"),
    postmenopausal_pct=("postmenopausal", "mean"),
    nodes_median=("pnodes", "median"),
    grade_III_pct=("tgrade", lambda s: (s == "III").mean()),
    tsize_median_mm=("tsize", "median"),
    pr_positive_pct=("pr_positive", "mean"),
).rename(index=ht_labels).T
display(table1.round(2))
display(pd.concat({name: km_summary(df.loc[df["postmenopausal"] == meno], "hormone_therapy", labels=ht_labels)
                   for meno, name in [(0, "Premenopausal"), (1, "Postmenopausal")]})[["n", "events", "RFS 5 y"]])

**Questions**
1. Were the groups balanced? Which characteristic differs the most between treated and untreated women?
2. Within each menopausal group, does hormone therapy still look beneficial? Is the size of the 5-year gain similar?
3. In premenopausal women the p-value is not significant. Is that "no effect" or "not enough evidence"? Count the treated premenopausal women before answering
4. Why can a variable that does not predict the outcome (Plot 3d) still matter when we compare treatments?

**Your notes**
-

## Plot 5. Everything at once: crude vs adjusted hazard ratios (Cox model)

**Place your bet:** after adjusting for all other factors, will the hormone therapy effect get weaker, stronger or stay the same?

**What the plot shows:** a forest plot with two estimates per variable
- **Crude HR** (grey): hazard ratio (HR) from a Cox model with that variable alone
- **Adjusted HR** (blue): HR from one multivariable Cox model with all variables together
- HR < 1 means lower risk, HR > 1 higher risk. If the 95% confidence interval (CI) crosses 1, the evidence is weak
- Units are chosen to be clinically meaningful: age per 10 years, tumour size per 10 mm, nodes per 5 extra nodes

The model is fitted on all 686 patients because the goal here is **explanation** (which factors matter). Plot 7 uses a train/test split because there the goal is **prediction**

In [ ]:
# Plot 5. Crude vs adjusted hazard ratios
X = pd.DataFrame({
    "hormone_therapy": df["hormone_therapy"],
    "age_per_10y": df["age"] / 10,
    "postmenopausal": df["postmenopausal"],
    "tsize_per_10mm": df["tsize"] / 10,
    "grade_II_vs_I": (df["tgrade"] == "II").astype(int),
    "grade_III_vs_I": (df["tgrade"] == "III").astype(int),
    "nodes_per_5": df["pnodes"] / 5,
    "pr_positive": df["pr_positive"],
    "er_positive": df["er_positive"],
})
covariates = list(X.columns)
cox_df = pd.concat([X, df[["years", "event"]]], axis=1)

# Adjusted model: all covariates together
cph = CoxPHFitter().fit(cox_df, duration_col="years", event_col="event")

# Crude models: one covariate at a time (grade I/II/III enters as one variable with two dummies)
crude = {}
for var in covariates:
    cols = ["grade_II_vs_I", "grade_III_vs_I"] if var.startswith("grade") else [var]
    m = CoxPHFitter().fit(cox_df[cols + ["years", "event"]], duration_col="years", event_col="event")
    crude[var] = m.summary.loc[var, ["exp(coef)", "exp(coef) lower 95%", "exp(coef) upper 95%"]].to_numpy()

adj = cph.summary[["exp(coef)", "exp(coef) lower 95%", "exp(coef) upper 95%", "p"]]
forest = pd.DataFrame({
    "crude HR": [crude[v][0] for v in covariates],
    "crude low": [crude[v][1] for v in covariates],
    "crude high": [crude[v][2] for v in covariates],
    "adjusted HR": adj["exp(coef)"].to_numpy(),
    "adj low": adj["exp(coef) lower 95%"].to_numpy(),
    "adj high": adj["exp(coef) upper 95%"].to_numpy(),
    "adjusted p": adj["p"].to_numpy(),
}, index=covariates)

fig, ax = plt.subplots(figsize=(10, 6.5))
y = np.arange(len(covariates))[::-1]
ax.errorbar(forest["crude HR"], y + 0.17,
            xerr=[forest["crude HR"] - forest["crude low"], forest["crude high"] - forest["crude HR"]],
            fmt="o", color=C["grey"], capsize=3, label="Crude (one variable at a time)")
ax.errorbar(forest["adjusted HR"], y - 0.17,
            xerr=[forest["adjusted HR"] - forest["adj low"], forest["adj high"] - forest["adjusted HR"]],
            fmt="s", color=C["blue"], capsize=3, label="Adjusted (all variables together)")
ax.axvline(1, color="black", linestyle="--", linewidth=1)
ax.set_xscale("log")
ax.set_xticks([0.3, 0.5, 0.7, 1, 1.5, 2, 3, 4])
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:g}"))
ax.xaxis.set_minor_formatter(plt.NullFormatter())
ax.set_xlim(0.3, 6)
ax.set_yticks(y)
ax.set_yticklabels(covariates)
ax.set_xlabel("Hazard ratio (log scale)   <- lower risk | higher risk ->")
ax.set_title(f"Crude vs adjusted hazard ratios  [model C-index = {cph.concordance_index_:.2f}]", fontsize=11)
ax.legend(loc="lower right", fontsize=9)
plt.tight_layout()
plt.show()

display(forest.round(3))

**Questions**
1. Was your bet right? What happens to the hormone therapy HR after adjustment? Turn the adjusted HR into a sentence: "hormone therapy is associated with a ...% lower risk of recurrence at any point in time"
2. Estrogen receptor (ER) status looks protective alone but not after adjustment. Which variable "absorbs" its effect, and why might that be? (Same story as SAPS-I vs SOFA in notebook 02)
3. Tumour size matters alone (crude HR per 10 mm) but much less after adjustment. Which variables might carry the same information?
4. Which three variables would you keep in a short risk tool? Use effect size and CI, not only p-values

**Your notes**
-

## Plot 6. Can we trust a single hazard ratio? (proportional hazards check)

**Place your bet:** will the extra risk of grade III tumours stay constant over the years, or fade?

**What the plot shows**
- The Cox model assumes each HR is **constant over time** (proportional hazards, PH)
- Log-minus-log plots, log(−log S(t)) vs log(time): **parallel curves** support PH, **converging or crossing curves** mean the effect changes over time
- The table gives the formal Schoenfeld residual test for every variable of the Plot 5 model (p < 0.05 flags a possible violation)

In [ ]:
# Plot 6. Proportional hazards diagnostics
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5), sharey=True)
grade_labels = {"I": "Grade I", "II": "Grade II", "III": "Grade III"}
grade_colors = {"I": C["sky"], "II": C["orange"], "III": C["red"]}

for ax, (col, labels, colors, title) in zip(axes, [
        ("tgrade", grade_labels, grade_colors, "(a) Tumour grade"),
        ("hormone_therapy", ht_labels, ht_colors, "(b) Hormone therapy")]):
    for g, lab in labels.items():
        sub = df.loc[df[col] == g]
        kmf = KaplanMeierFitter().fit(sub["years"], sub["event"])
        sf = kmf.survival_function_.iloc[:, 0]
        sf = sf[(sf > 0) & (sf < 1) & (sf.index > 0)]
        ax.step(np.log(sf.index), np.log(-np.log(sf)), where="post", color=colors[g], linewidth=2, label=lab)
    ax.set_title(f"{title}: log-minus-log curves", fontsize=11)
    ax.set_xlabel("log(years since surgery)")
    ax.set_ylabel("log(-log(RFS))")
    ax.legend(loc="lower right")
    # Mark 1, 3 and 5 years on the log axis
    for h in HORIZONS:
        ax.axvline(np.log(h), color="lightgrey", linestyle=":")
        ax.text(np.log(h), 0.97, f"{h} y", ha="center", fontsize=9, color=C["grey"],
                transform=ax.get_xaxis_transform())
plt.tight_layout()
plt.show()

ph = proportional_hazard_test(cph, cox_df, time_transform="rank").summary[["test_statistic", "p"]]
ph["PH violated? (p < 0.05)"] = np.where(ph["p"] < 0.05, "check", "ok")
display(ph.round(3))

**Questions**
1. Look at panel (a): do the grade III and grade II/I curves stay parallel, converge or cross? What does that say about **when** grade matters most?
2. Panel (b): is the hormone therapy effect roughly constant over time? Does that make the HR from Plot 5 more or less trustworthy?
3. Which variables does the formal test flag? Does the test agree with your eyes?
4. If a tumour board member asks "how much worse is grade III?", what is a more honest answer than one single HR?

**Your notes**
-

## Plot 7. From model to bedside: risk groups and patient profiles

**Place your bet:** if the same hormone therapy HR applies to everyone, which patient gains more years recurrence-free from it, a low-risk or a high-risk patient?

**What the plot shows**
- Left: we fit the Cox model on 70% of patients (train) and split the other 30% (test) into low, medium and high predicted risk, using cut-offs from the training set (no leakage). The C-index tells how well the model ranks patients it has not seen
- Right: predicted RFS curves for three patient profiles, **without** (solid) and **with** (dashed) hormone therapy, using the Plot 5 model

In [ ]:
# Plot 7. (a) Risk groups on unseen patients, (b) individual predictions for three profiles

# (a) Train/test split, cut-offs from TRAIN only
train, test = train_test_split(cox_df, test_size=0.30, random_state=RANDOM_STATE, stratify=cox_df["event"])
cph_train = CoxPHFitter().fit(train, duration_col="years", event_col="event")
risk_train = cph_train.predict_partial_hazard(train)
cuts = np.quantile(risk_train, [1 / 3, 2 / 3])
test = test.assign(risk_group=pd.cut(cph_train.predict_partial_hazard(test), bins=[-np.inf, *cuts, np.inf],
                                     labels=["Low", "Medium", "High"]))
c_train = cph_train.concordance_index_
c_test = concordance_index(test["years"], -cph_train.predict_partial_hazard(test), test["event"])

# (b) Three patient profiles
profiles = pd.DataFrame([
    {"name": "Ana: 42 y, premenopausal, 15 mm, grade I, 1 node, PR+ ER+",
     "age_per_10y": 4.2, "postmenopausal": 0, "tsize_per_10mm": 1.5, "grade_II_vs_I": 0, "grade_III_vs_I": 0,
     "nodes_per_5": 1 / 5, "pr_positive": 1, "er_positive": 1},
    {"name": "Bea: 55 y, postmenopausal, 25 mm, grade II, 4 nodes, PR+ ER+",
     "age_per_10y": 5.5, "postmenopausal": 1, "tsize_per_10mm": 2.5, "grade_II_vs_I": 1, "grade_III_vs_I": 0,
     "nodes_per_5": 4 / 5, "pr_positive": 1, "er_positive": 1},
    {"name": "Carmen: 63 y, postmenopausal, 40 mm, grade III, 12 nodes, PR- ER-",
     "age_per_10y": 6.3, "postmenopausal": 1, "tsize_per_10mm": 4.0, "grade_II_vs_I": 0, "grade_III_vs_I": 1,
     "nodes_per_5": 12 / 5, "pr_positive": 0, "er_positive": 0},
]).set_index("name")
prof_colors = [C["sky"], C["orange"], C["red"]]
times = np.linspace(0, T_MAX, 300)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
km_panel(axes[0], test, "risk_group", labels={"Low": "Low risk", "Medium": "Medium risk", "High": "High risk"},
         colors={"Low": C["sky"], "Medium": C["orange"], "High": C["red"]},
         title=f"(a) Test set by predicted risk, C-index train {c_train:.2f} / test {c_test:.2f}", at_risk=True)

rows = []
for (name, prof), col in zip(profiles.iterrows(), prof_colors):
    for ht, ls in [(0, "-"), (1, "--")]:
        x = prof.to_frame().T.assign(hormone_therapy=ht)[covariates].astype(float)
        sf = cph.predict_survival_function(x, times=times).iloc[:, 0]
        axes[1].plot(times, sf, linestyle=ls, color=col, linewidth=2,
                     label=name.split(":")[0] + (" + hormone therapy" if ht else ""))
    s5 = {ht: float(cph.predict_survival_function(prof.to_frame().T.assign(hormone_therapy=ht)[covariates].astype(float),
                                                  times=[5]).iloc[0, 0]) for ht in [0, 1]}
    rows.append({"profile": name, "RFS 5 y without": s5[0], "RFS 5 y with": s5[1],
                 "absolute gain": s5[1] - s5[0], "NNT at 5 y": 1 / (s5[1] - s5[0])})
axes[1].axvline(5, color="lightgrey", linestyle=":")
axes[1].set_xlim(0, T_MAX)
axes[1].set_ylim(0, 1.02)
axes[1].set_xlabel("Years since surgery")
axes[1].set_ylabel("Predicted recurrence-free survival")
axes[1].set_title("(b) Three patient profiles: solid = no hormone therapy, dashed = hormone therapy", fontsize=11)
axes[1].legend(loc="lower left", fontsize=9)
plt.tight_layout()
plt.show()

display(pd.DataFrame(rows).set_index("profile").round(3))

**Questions**
1. Panel (a): how well does the model separate unseen patients? Is the C-index on the test set close to the training one? What does that tell you about overfitting?
2. Panel (b): who gains most from hormone therapy in absolute terms, and why, if the HR is the same for everyone?
3. **Clinical red flag:** Carmen has a receptor-negative tumour (PR− and ER−). Hormone therapy works by blocking hormone receptors. Should we trust the dashed red curve? What did the model assume that may be wrong?
4. How would you use panel (a) in the follow-up clinic? What would you need to check before using it with real patients?

**Your notes**
-

## Your turn: test or challenge your findings (optional, if time allows)

Pick **one or two** ideas, run them, and decide whether they confirm or contradict your conclusions. The helpers `quick_km(...)` and `quick_cox(...)` make this a one-liner. Some ideas:

- **Age bands:** are very young women (< 40 years) a high-risk group? `quick_km("age", bins=[0, 40, 50, 60, 100])`
- **Tumour size:** `quick_km("tsize", bins=[0, 20, 30, 200], labels=["≤20 mm", "21-30 mm", ">30 mm"])`
- **ER status alone:** `quick_km("er_positive")`, then compare with Plot 5
- **Does hormone therapy only work in receptor-positive tumours?** (the Carmen question) Run KM by hormone therapy within PR+ and PR− women, or add an interaction term to the Cox model (starter code below)
- **Stratified Cox for grade** (remedy for a PH violation, notebook 02): `quick_cox([...], strata=["tgrade"])`
- **RMST at another horizon:** does the hormone therapy gain look different at 3 years? Use `km_summary(df, "hormone_therapy", labels=ht_labels, t_rmst=3)`
- **Your own idea:** any plot that would convince the Skeptic

In [ ]:
# Starter: does the hormone therapy effect depend on PR status?
for pr, name in [(1, "PR positive"), (0, "PR negative")]:
    sub = df.loc[df["pr_positive"] == pr]
    fig, ax = plt.subplots(figsize=(7, 5))
    km_panel(ax, sub, "hormone_therapy", labels=ht_labels, colors=ht_colors, title=f"Hormone therapy, {name} only")
    plt.tight_layout()
    plt.show()

# Interaction term in the Cox model: HR of hormone therapy in PR- = exp(hormone_therapy),
# in PR+ = exp(hormone_therapy + ht_x_pr)
cox_int = cox_df.assign(ht_x_pr=cox_df["hormone_therapy"] * cox_df["pr_positive"])
m_int = quick_cox(covariates + ["ht_x_pr"], data=cox_int)

In [ ]:
# Your exploration here

In [ ]:
# More exploration (add as many cells as you need)

## Tumour board brief (5 minutes, write it together)

Fill in the template. Every claim must point to a plot (e.g. "Plot 5"). Be ready to present it in 60 seconds

**1. Three findings** (what the data show)
-
-
-

**2. Two implications** (what this means for patients and for the unit)
-
-

**3. Two recommendations** (for clinicians, patients or managers: who should do what)
-
-

**4. One caveat** (what could make us wrong, and what we would check next)
-

**5. Key takeaway** (one sentence, under 20 words, active voice, something you would say out loud)
-